# Wine Quality Prediction Using Machine Learning
## Notebook tạo bảng và hình cho bài báo theo cấu trúc Springer

Notebook này đọc hai file UCI `winequality-red.csv` và `winequality-white.csv`, chạy các thí nghiệm có thể tái lập và xuất bảng/hình vào thư mục `outputs/`.

**Lưu ý:** Các bảng kết quả mô hình chỉ được tạo sau khi chạy code. Không chép số liệu minh họa vào bài báo như kết quả thực nghiệm. Dataset gốc dùng dấu chấm phẩy (`;`) để phân tách cột; notebook tự thử đọc đúng định dạng.


In [1]:
# Nếu chạy trên Google Colab, bỏ comment dòng dưới nếu thiếu thư viện:
# %pip install -q imbalanced-learn

import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import SelectKBest, f_classif

RANDOM_STATE = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", None)


ModuleNotFoundError: No module named 'sklearn'

## 1. Đọc hai bộ dữ liệu

Đặt hai file CSV cùng thư mục với notebook. Nếu chạy Colab, tải chúng lên phiên làm việc trước. Không gộp cột `type` vào 11 đặc trưng khi phân tích riêng Red và White; cột này chỉ được thêm để phân tích dữ liệu gộp nếu cần.


In [ ]:
def load_uci_csv(path):
    # UCI Wine Quality CSV thường dùng dấu ; làm delimiter
    df = pd.read_csv(path, sep=";")
    if df.shape[1] == 1:  # dự phòng nếu delimiter khác
        df = pd.read_csv(path, sep=None, engine="python")
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    return df

RED_PATH = Path("winequality-red.csv")
WHITE_PATH = Path("winequality-white.csv")

if not RED_PATH.exists() or not WHITE_PATH.exists():
    try:
        from google.colab import files
        print("Chọn tải lên cả winequality-red.csv và winequality-white.csv")
        uploaded = files.upload()
    except Exception:
        raise FileNotFoundError(
            "Không tìm thấy CSV. Hãy đặt winequality-red.csv và "
            "winequality-white.csv cạnh notebook hoặc tải lên Colab."
        )

red = load_uci_csv(RED_PATH)
white = load_uci_csv(WHITE_PATH)
red["wine_type"] = "Red"
white["wine_type"] = "White"
combined = pd.concat([red, white], ignore_index=True)

print("Red shape:", red.shape)
print("White shape:", white.shape)
print("Combined shape:", combined.shape)
display(combined.head())


In [ ]:
# Kiểm tra dữ liệu và tạo bảng mô tả
target = "quality"
feature_cols = [c for c in red.columns if c != target]

dataset_summary = pd.DataFrame([
    {"Dataset": "Red wine", "Samples": len(red), "Features": len(feature_cols),
     "Missing cells": int(red[feature_cols + [target]].isna().sum().sum()),
     "Quality classes": int(red[target].nunique()),
     "Minimum quality": int(red[target].min()), "Maximum quality": int(red[target].max())},
    {"Dataset": "White wine", "Samples": len(white), "Features": len(feature_cols),
     "Missing cells": int(white[feature_cols + [target]].isna().sum().sum()),
     "Quality classes": int(white[target].nunique()),
     "Minimum quality": int(white[target].min()), "Maximum quality": int(white[target].max())},
    {"Dataset": "Combined", "Samples": len(combined), "Features": len(feature_cols),
     "Missing cells": int(combined[feature_cols + [target]].isna().sum().sum()),
     "Quality classes": int(combined[target].nunique()),
     "Minimum quality": int(combined[target].min()), "Maximum quality": int(combined[target].max())}
])
display(dataset_summary)
dataset_summary.to_csv(OUTPUT_DIR / "table_dataset_summary.csv", index=False)


## 2. Hình dữ liệu đầu vào

Các hình dưới đây có thể dùng trong phần Dataset Description/Exploratory Data Analysis. Caption cần mô tả nội dung và nguồn dữ liệu UCI; không diễn giải tương quan thành quan hệ nhân quả.


In [ ]:
# Figure 1: phân bố nhãn chất lượng ở Red và White
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)
for ax, df, label in zip(axes, [red, white], ["Red wine", "White wine"]):
    counts = df[target].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values)
    ax.set_title(label)
    ax.set_xlabel("Quality score")
    ax.set_ylabel("Number of samples")
    ax.grid(axis="y", alpha=0.25)
fig.suptitle("Quality-class distribution in the UCI Wine Quality datasets")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fig1_quality_distribution.png", bbox_inches="tight")
plt.show()

# Bảng phân bố lớp và phần trăm
class_dist = pd.concat([
    red[target].value_counts().sort_index().rename("Red_count"),
    white[target].value_counts().sort_index().rename("White_count")
], axis=1).fillna(0).astype(int).reset_index(names="Quality")
class_dist["Red_percent"] = (class_dist["Red_count"] / len(red) * 100).round(2)
class_dist["White_percent"] = (class_dist["White_count"] / len(white) * 100).round(2)
display(class_dist)
class_dist.to_csv(OUTPUT_DIR / "table_quality_distribution.csv", index=False)


In [ ]:
# Figure 2: ma trận tương quan của các đặc trưng và quality (Red + White riêng)
for df, label, filename in [
    (red, "Red wine", "fig2_red_correlation.png"),
    (white, "White wine", "fig3_white_correlation.png")
]:
    corr = df[feature_cols + [target]].corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(corr.values, aspect="auto", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=90)
    ax.set_yticks(range(len(corr.index)), corr.index)
    ax.set_title(f"Feature correlation matrix — {label}")
    fig.colorbar(im, ax=ax, label="Pearson correlation")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, bbox_inches="tight")
    plt.show()


## 3. Thiết kế thí nghiệm và so sánh mô hình

Phần này sử dụng **Stratified 5-Fold Cross-Validation** để so sánh DT, RF, SVM và KNN. `StandardScaler` nằm trong pipeline cho SVM/KNN để tránh rò rỉ dữ liệu giữa các folds. DT/RF được đánh giá trên dữ liệu gốc làm baseline. Các metric được tính theo từng fold rồi tổng hợp bằng trung bình và độ lệch chuẩn.


In [ ]:
X = red[feature_cols].copy()
y = red[target].astype(int).copy()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
models = {
    "Decision Tree": SkPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "Random Forest": SkPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE,
                                         class_weight=None, n_jobs=-1))
    ]),
    "SVM (scaled)": SkPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", random_state=RANDOM_STATE))
    ]),
    "KNN (scaled)": SkPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ])
}
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "macro_f1": "f1_macro",
    "macro_precision": "precision_macro",
    "macro_recall": "recall_macro"
}

rows = []
for name, model in models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1,
                            return_train_score=False)
    row = {"Model": name}
    for metric in scoring:
        vals = scores[f"test_{metric}"]
        row[f"{metric}_mean"] = vals.mean()
        row[f"{metric}_std"] = vals.std()
    rows.append(row)

model_results = pd.DataFrame(rows).sort_values("macro_f1_mean", ascending=False)
display(model_results.round(4))
model_results.to_csv(OUTPUT_DIR / "table_model_comparison.csv", index=False)

# Figure 4: so sánh Macro-F1
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(model_results["Model"], model_results["macro_f1_mean"],
       yerr=model_results["macro_f1_std"], capsize=4)
ax.set_ylabel("Macro-F1 (mean ± standard deviation)")
ax.set_title("Five-fold cross-validation: model comparison")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fig4_model_comparison.png", bbox_inches="tight")
plt.show()


## 4. Thí nghiệm Raw vs StandardScaler

Để so sánh công bằng, dùng cùng các folds và cùng mô hình. Ví dụ dưới đây dùng SVM và KNN — hai mô hình thường nhạy với thang đo. Bảng này là thí nghiệm của notebook, không phải kết quả trích từ bài báo.


In [ ]:
scale_rows = []
scale_models = {
    "SVM - Raw": SkPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", SVC(kernel="rbf", random_state=RANDOM_STATE))
    ]),
    "SVM - StandardScaler": SkPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", random_state=RANDOM_STATE))
    ]),
    "KNN - Raw": SkPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),
    "KNN - StandardScaler": SkPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ])
}
for name, model in scale_models.items():
    scores = cross_validate(model, X, y, cv=cv,
                            scoring={"accuracy":"accuracy","balanced_accuracy":"balanced_accuracy","macro_f1":"f1_macro"},
                            n_jobs=-1)
    scale_rows.append({
        "Experiment": name,
        "Accuracy_mean": scores["test_accuracy"].mean(),
        "Balanced_Accuracy_mean": scores["test_balanced_accuracy"].mean(),
        "Macro_F1_mean": scores["test_macro_f1"].mean()
    })
scaling_results = pd.DataFrame(scale_rows)
display(scaling_results.round(4))
scaling_results.to_csv(OUTPUT_DIR / "table_scaling_ablation.csv", index=False)


## 5. SMOTE ablation (tùy chọn)

SMOTE chỉ được áp dụng bên trong từng tập huấn luyện của cross-validation bằng `imblearn.pipeline.Pipeline`; không resample toàn bộ dữ liệu trước khi chia folds. Nếu môi trường chưa cài `imbalanced-learn`, chạy lệnh cài đặt ở đầu notebook.


In [ ]:
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE

    smote_rows = []
    smote_estimators = {
        "RF - No SMOTE": SkPipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1))
        ]),
        "RF - SMOTE": ImbPipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=3)),
            ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1))
        ])
    }
    for name, model in smote_estimators.items():
        scores = cross_validate(model, X, y, cv=cv,
                                scoring={"accuracy":"accuracy","balanced_accuracy":"balanced_accuracy","macro_f1":"f1_macro","macro_recall":"recall_macro"},
                                n_jobs=-1)
        smote_rows.append({
            "Experiment": name,
            "Accuracy_mean": scores["test_accuracy"].mean(),
            "Balanced_Accuracy_mean": scores["test_balanced_accuracy"].mean(),
            "Macro_F1_mean": scores["test_macro_f1"].mean(),
            "Macro_Recall_mean": scores["test_macro_recall"].mean()
        })
    smote_results = pd.DataFrame(smote_rows)
    display(smote_results.round(4))
    smote_results.to_csv(OUTPUT_DIR / "table_smote_ablation.csv", index=False)
except ImportError:
    print("Chưa có imbalanced-learn. Cài bằng: %pip install -q imbalanced-learn rồi chạy lại cell này.")


## 6. Feature Importance và Confusion Matrix

Dùng tập train/test phân tầng để tạo confusion matrix và tính feature importance. Feature importance từ Random Forest là mức đóng góp theo tiêu chí chia nhánh của mô hình; permutation importance đo mức giảm metric khi xáo trộn một đặc trưng. Đây là hai định nghĩa khác nhau nên cần ghi rõ phương pháp trong bài báo.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

rf = SkPipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1))
])
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

test_metrics = pd.DataFrame([{
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
    "Macro Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "Macro Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "Macro-F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "Weighted-F1": f1_score(y_test, y_pred, average="weighted", zero_division=0)
}])
display(test_metrics.round(4))
test_metrics.to_csv(OUTPUT_DIR / "table_holdout_metrics.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Random Forest confusion matrix — red wine hold-out set")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fig5_confusion_matrix_rf.png", bbox_inches="tight")
plt.show()

rf_model = rf.named_steps["model"]
rf_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)
display(rf_importance.round(5))
rf_importance.to_csv(OUTPUT_DIR / "table_rf_feature_importance.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
plot_imp = rf_importance.sort_values("Importance", ascending=True)
ax.barh(plot_imp["Feature"], plot_imp["Importance"])
ax.set_xlabel("Mean decrease in impurity")
ax.set_title("Random Forest feature importance — red wine")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fig6_rf_feature_importance.png", bbox_inches="tight")
plt.show()

perm = permutation_importance(rf, X_test, y_test, n_repeats=10,
                              random_state=RANDOM_STATE, scoring="f1_macro", n_jobs=-1)
perm_imp = pd.DataFrame({
    "Feature": feature_cols,
    "Importance_mean": perm.importances_mean,
    "Importance_std": perm.importances_std
}).sort_values("Importance_mean", ascending=False)
display(perm_imp.round(5))
perm_imp.to_csv(OUTPUT_DIR / "table_permutation_importance.csv", index=False)


## 7. So sánh riêng Red và White

Dùng cùng mô hình và cùng giao thức cross-validation trên từng bộ dữ liệu. Vì số lượng mẫu và phân bố lớp khác nhau, phần thảo luận cần lưu ý đây không phải thí nghiệm trên cùng các quan sát.


In [ ]:
def evaluate_dataset(df, label):
    Xd = df[feature_cols].copy()
    yd = df[target].astype(int).copy()
    cv_local = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    estimator = SkPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1))
    ])
    scores = cross_validate(estimator, Xd, yd, cv=cv_local,
                            scoring={"accuracy":"accuracy","balanced_accuracy":"balanced_accuracy","macro_f1":"f1_macro"},
                            n_jobs=-1)
    return {
        "Dataset": label, "Samples": len(df),
        "Accuracy_mean": scores["test_accuracy"].mean(),
        "Balanced_Accuracy_mean": scores["test_balanced_accuracy"].mean(),
        "Macro_F1_mean": scores["test_macro_f1"].mean()
    }

red_white_results = pd.DataFrame([
    evaluate_dataset(red, "Red wine"),
    evaluate_dataset(white, "White wine")
])
display(red_white_results.round(4))
red_white_results.to_csv(OUTPUT_DIR / "table_red_vs_white.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(red_white_results))
width = 0.25
for i, metric in enumerate(["Accuracy_mean", "Balanced_Accuracy_mean", "Macro_F1_mean"]):
    ax.bar(x + (i-1)*width, red_white_results[metric], width, label=metric.replace("_mean",""))
ax.set_xticks(x, red_white_results["Dataset"])
ax.set_ylim(0, 1)
ax.set_ylabel("Cross-validation score")
ax.set_title("Random Forest performance: red vs white wine")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fig7_red_vs_white.png", bbox_inches="tight")
plt.show()


## 8. Xuất danh sách file đã tạo

Các file CSV là bảng dữ liệu; các PNG là hình có thể chèn vào Word. Trong bài báo, đánh số lại hình/bảng theo thứ tự xuất hiện và viết caption theo phong cách mẫu Springer (ví dụ: `Fig. 1. Quality-class distribution in the UCI Wine Quality datasets.`). Không bắt buộc phải giữ nguyên mọi hình; chỉ chọn hình hỗ trợ lập luận.


In [ ]:
print("Các tệp được tạo trong:", OUTPUT_DIR.resolve())
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f.name, "-", f.stat().st_size, "bytes")
